# Proyecto Big Data - Steam Reviews (Spark)


In [1]:
# ==========================================================
# CONFIGURACION DE RECOLECCION DE TIEMPOS
# ==========================================================
# Arquitectura A: 1 Master + 2 Workers (n2-standard-4)

tiempos_resultados = {}
arquitectura = "2_workers"


In [2]:
!pip install pyspark

  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.9
    Uninstalling py4j-0.10.9.9:
      Successfully uninstalled py4j-0.10.9.9


In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SteamReviews") \
    .getOrCreate()

dfs = spark.read.csv(
    "gs://bigdata-2026-02/proyecto01/steam_reviews_500k.csv",
    header=True,
    inferSchema=True,
    multiLine=True,
    escape='"',
    quote='"'
)

dfs.show(5)
dfs.printSchema()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/21 09:34:08 INFO SparkEnv: Registering MapOutputTracker
26/09/21 09:34:08 INFO SparkEnv: Registering BlockManagerMaster
26/09/21 09:34:08 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/09/21 09:34:08 INFO SparkEnv: Registering OutputCommitCoordinator


+----------------+-------+-----------------+-----------------+----------------------+------------------+-----------------------+------------------------------+-------------------------+------------------+---------+--------------------+-----------------+-----------------+--------+--------+-----------+-------------------+-------------+--------------+-----------------+---------------------------+---------------------+--------------------+
|recommendationid|  appid|             game|   author_steamid|author_num_games_owned|author_num_reviews|author_playtime_forever|author_playtime_last_two_weeks|author_playtime_at_review|author_last_played| language|              review|timestamp_created|timestamp_updated|voted_up|votes_up|votes_funny|weighted_vote_score|comment_count|steam_purchase|received_for_free|written_during_early_access|hidden_in_steam_china|steam_china_location|
+----------------+-------+-----------------+-----------------+----------------------+------------------+----------------

## Consulta 1 - Exploración y validación del dataset

Objetivo: conocer dimensiones, estructura, tipos de datos y valores nulos del dataset.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [4]:
# ==========================================================
# CONSULTA 1 - SPARK
# ==========================================================

import time

from pyspark.sql.functions import col, sum


inicio = time.time()

print("===== SPARK =====")


print("\nDimensiones del dataset")


filas = dfs.count()

print("Filas:", filas)

print("Columnas:", len(dfs.columns))


print("\nEstructura de datos")

dfs.printSchema()


print("\nValores nulos")


dfs.select(
    [
        sum(
            col(c)
            .isNull()
            .cast("int")
        )
        .alias(c)

        for c in dfs.columns
    ]
).show()


fin = time.time()

print("\nTiempo de ejecución:",
      fin - inicio,
      "segundos")
tiempos_resultados["Spark_Consulta_1"] = fin - inicio


===== SPARK =====

Dimensiones del dataset


Filas: 500000
Columnas: 24

Estructura de datos
root
 |-- recommendationid: integer (nullable = true)
 |-- appid: integer (nullable = true)
 |-- game: string (nullable = true)
 |-- author_steamid: long (nullable = true)
 |-- author_num_games_owned: integer (nullable = true)
 |-- author_num_reviews: integer (nullable = true)
 |-- author_playtime_forever: integer (nullable = true)
 |-- author_playtime_last_two_weeks: integer (nullable = true)
 |-- author_playtime_at_review: integer (nullable = true)
 |-- author_last_played: integer (nullable = true)
 |-- language: string (nullable = true)
 |-- review: string (nullable = true)
 |-- timestamp_created: integer (nullable = true)
 |-- timestamp_updated: integer (nullable = true)
 |-- voted_up: integer (nullable = true)
 |-- votes_up: integer (nullable = true)
 |-- votes_funny: long (nullable = true)
 |-- weighted_vote_score: double (nullable = true)
 |-- comment_count: integer (nullable = true)
 |-- steam_purchase: integer (nullable = true)
 

+----------------+-----+----+--------------+----------------------+------------------+-----------------------+------------------------------+-------------------------+------------------+--------+------+-----------------+-----------------+--------+--------+-----------+-------------------+-------------+--------------+-----------------+---------------------------+---------------------+--------------------+
|recommendationid|appid|game|author_steamid|author_num_games_owned|author_num_reviews|author_playtime_forever|author_playtime_last_two_weeks|author_playtime_at_review|author_last_played|language|review|timestamp_created|timestamp_updated|voted_up|votes_up|votes_funny|weighted_vote_score|comment_count|steam_purchase|received_for_free|written_during_early_access|hidden_in_steam_china|steam_china_location|
+----------------+-----+----+--------------+----------------------+------------------+-----------------------+------------------------------+-------------------------+------------------+

## Consulta 2 - Eliminación de duplicados

Objetivo: eliminar registros repetidos considerando author_steamid, appid y review.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [5]:
# ==========================================================
# CONSULTA 2 - ELIMINACIÓN DE DUPLICADOS CON SPARK
# ==========================================================

import time

inicio = time.time()

print("===== SPARK =====")


# Registros iniciales

registros_iniciales = dfs.count()


print("Registros iniciales:",
      registros_iniciales)


# Eliminación de duplicados

dfs_clean = dfs.dropDuplicates(
    [
        "author_steamid",
        "appid",
        "review"
    ]
)


# Acción para ejecutar

registros_finales = dfs_clean.count()


print("Registros después de eliminar duplicados:",
      registros_finales)


print("Duplicados eliminados:",
      registros_iniciales - registros_finales)


fin = time.time()

print("\nTiempo de ejecución:",
      fin - inicio,
      "segundos")
tiempos_resultados["Spark_Consulta_2"] = fin - inicio


===== SPARK =====


Registros iniciales: 500000


Registros después de eliminar duplicados: 315809
Duplicados eliminados: 184191

Tiempo de ejecución: 27.44123077392578 segundos


## Consulta 3 - Tratamiento de valores nulos

Objetivo: identificar valores faltantes y limpiar registros sin información en review.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [6]:
# ==========================================================
# CONSULTA 3 - TRATAMIENTO DE VALORES NULOS CON SPARK
# ==========================================================

import time

from pyspark.sql.functions import col, sum


inicio = time.time()

print("===== SPARK =====")


registros_iniciales = dfs_clean.count()


print("Registros iniciales:",
      registros_iniciales)


print("\nValores nulos antes:")


dfs_clean.select(
    [
        sum(
            col(c)
            .isNull()
            .cast("int")
        )
        .alias(c)

        for c in dfs_clean.columns
    ]
).show()



# Eliminación de registros sin review

dfs_null_clean = dfs_clean.dropna(
    subset=["review"]
)



registros_finales = dfs_null_clean.count()


print("Registros después de limpiar:",
      registros_finales)


print("Registros eliminados:",
      registros_iniciales - registros_finales)



print("\nValores nulos después:")


dfs_null_clean.select(
    [
        sum(
            col(c)
            .isNull()
            .cast("int")
        )
        .alias(c)

        for c in dfs_null_clean.columns
    ]
).show()



fin = time.time()

print("\nTiempo de ejecución:",
      fin - inicio,
      "segundos")
tiempos_resultados["Spark_Consulta_3"] = fin - inicio


===== SPARK =====


Registros iniciales: 315809

Valores nulos antes:


26/09/21 09:35:51 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----------------+-----+----+--------------+----------------------+------------------+-----------------------+------------------------------+-------------------------+------------------+--------+------+-----------------+-----------------+--------+--------+-----------+-------------------+-------------+--------------+-----------------+---------------------------+---------------------+--------------------+
|recommendationid|appid|game|author_steamid|author_num_games_owned|author_num_reviews|author_playtime_forever|author_playtime_last_two_weeks|author_playtime_at_review|author_last_played|language|review|timestamp_created|timestamp_updated|voted_up|votes_up|votes_funny|weighted_vote_score|comment_count|steam_purchase|received_for_free|written_during_early_access|hidden_in_steam_china|steam_china_location|
+----------------+-----+----+--------------+----------------------+------------------+-----------------------+------------------------------+-------------------------+------------------+

Registros después de limpiar: 315809
Registros eliminados: 0

Valores nulos después:


+----------------+-----+----+--------------+----------------------+------------------+-----------------------+------------------------------+-------------------------+------------------+--------+------+-----------------+-----------------+--------+--------+-----------+-------------------+-------------+--------------+-----------------+---------------------------+---------------------+--------------------+
|recommendationid|appid|game|author_steamid|author_num_games_owned|author_num_reviews|author_playtime_forever|author_playtime_last_two_weeks|author_playtime_at_review|author_last_played|language|review|timestamp_created|timestamp_updated|voted_up|votes_up|votes_funny|weighted_vote_score|comment_count|steam_purchase|received_for_free|written_during_early_access|hidden_in_steam_china|steam_china_location|
+----------------+-----+----+--------------+----------------------+------------------+-----------------------+------------------------------+-------------------------+------------------+

## Consulta 4 - Transformación de variables

Objetivo: crear review_length como cantidad de caracteres de cada reseña.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [7]:
# ==========================================================
# CONSULTA 4 - TRANSFORMACIÓN DE VARIABLES CON SPARK
# ==========================================================

import time

from pyspark.sql.functions import length

inicio = time.time()

print("===== SPARK =====")


# Crear nueva columna

dfs_transform = dfs_null_clean.withColumn(
    "review_length",
    length("review")
)


# Acción para ejecutar Spark

registros = dfs_transform.count()


print("Registros procesados:",
      registros)


print("\nEjemplo de transformación:")

dfs_transform.select(
    [
        "review",
        "review_length"
    ]
).show(5)


fin = time.time()

print("\nTiempo de ejecución:",
      fin - inicio,
      "segundos")
tiempos_resultados["Spark_Consulta_4"] = fin - inicio


===== SPARK =====


Registros procesados: 315809

Ejemplo de transformación:


+-------------------------------------+-------------+
|                               review|review_length|
+-------------------------------------+-------------+
|                 There's a lot to ...|         1107|
|            感觉盗版比正版还多。。。 |           13|
|                 💢Operam !💢 📌 С...|         7820|
|        你这辈子就是被Terraria害了...|          362|
|一个魂们，我们给这游戏整个大奖提名...|           22|
+-------------------------------------+-------------+
only showing top 5 rows


Tiempo de ejecución: 32.51998782157898 segundos


## Consulta 5 - Filtrado de reseñas recomendadas

Objetivo: seleccionar registros donde voted_up sea igual a 1.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [8]:
# ==========================================================
# CONSULTA 5 - FILTRADO DE RESEÑAS RECOMENDADAS CON SPARK
# ==========================================================

import time

from pyspark.sql.functions import col


inicio = time.time()

print("===== SPARK =====")


registros_iniciales = dfs_transform.count()


print("Registros iniciales:",
      registros_iniciales)



# Filtrar reseñas recomendadas

dfs_positive = dfs_transform.filter(
    col("voted_up") == 1
)


registros_finales = dfs_positive.count()


print("Registros recomendados:",
      registros_finales)


print("Porcentaje de recomendaciones:",
      (registros_finales / registros_iniciales) * 100,
      "%")


print("\nEjemplo de datos:")

dfs_positive.select(
    [
        "game",
        "review",
        "voted_up"
    ]
).show(5)



fin = time.time()

print("\nTiempo de ejecución:",
      fin - inicio,
      "segundos")
tiempos_resultados["Spark_Consulta_5"] = fin - inicio


===== SPARK =====


Registros iniciales: 315809


Registros recomendados: 266640
Porcentaje de recomendaciones: 84.43077936347603 %

Ejemplo de datos:


+---------------+--------------------+--------+
|           game|              review|voted_up|
+---------------+--------------------+--------+
|    War Thunder|    玩了一会先睡一会|       1|
|          SMITE|        就玩了一小会|       1|
|Bless Unleashed|        就玩了一小会|       1|
|   Satisfactory|Did you like & pl...|       1|
|    War Thunder| It has the A-10 now|       1|
+---------------+--------------------+--------+
only showing top 5 rows


Tiempo de ejecución: 46.27542567253113 segundos


## Consulta 6 - Cantidad de reseñas por videojuego

Objetivo: agrupar por game y calcular la cantidad total de reseñas.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [9]:
# ==========================================================
# CONSULTA 6 - CANTIDAD DE RESEÑAS POR VIDEOJUEGO - SPARK
# ==========================================================

import time

from pyspark.sql.functions import count, desc


inicio = time.time()

print("===== SPARK =====")


dfs_game_reviews = (
    dfs_transform
    .groupBy("game")
    .agg(
        count("*")
        .alias("total_reviews")
    )
    .orderBy(
        desc("total_reviews")
    )
)


# Acción para ejecutar Spark

cantidad_juegos = dfs_game_reviews.count()


print("Cantidad de videojuegos procesados:",
      cantidad_juegos)


print("\nTop videojuegos por cantidad de reseñas:")

dfs_game_reviews.show(10)



fin = time.time()

print("\nTiempo de ejecución:",
      fin - inicio,
      "segundos")
tiempos_resultados["Spark_Consulta_6"] = fin - inicio


===== SPARK =====


Cantidad de videojuegos procesados: 22992

Top videojuegos por cantidad de reseñas:


+--------------------+-------------+
|                game|total_reviews|
+--------------------+-------------+
|    Counter-Strike 2|         1923|
| PUBG: BATTLEGROUNDS|         1513|
|      Stardew Valley|         1413|
|            Terraria|         1264|
|  Grand Theft Auto V|         1228|
|The Witcher 3: Wi...|         1228|
|Tom Clancy's Rain...|         1224|
|          The Forest|         1168|
|    Wallpaper Engine|         1093|
|                Rust|         1034|
+--------------------+-------------+
only showing top 10 rows


Tiempo de ejecución: 34.41811418533325 segundos


## Consulta 7 - Porcentaje de recomendación por videojuego

Objetivo: calcular porcentaje de reseñas positivas por juego usando voted_up.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [10]:
# ==========================================================
# CONSULTA 7 - PORCENTAJE DE RECOMENDACIÓN POR VIDEOJUEGO
# SPARK
# ==========================================================

import time

from pyspark.sql.functions import (
    count,
    sum,
    col,
    desc
)


inicio = time.time()

print("===== SPARK =====")


dfs_recommendation = (
    dfs_transform
    .groupBy("game")
    .agg(
        count("*")
        .alias("total_reviews"),

        sum("voted_up")
        .alias("positive_reviews")
    )
    .withColumn(
        "recommendation_percentage",
        (
            col("positive_reviews") /
            col("total_reviews")
            * 100
        )
    )
    .orderBy(
        desc("recommendation_percentage")
    )
)


# Ejecuta Spark

cantidad = dfs_recommendation.count()


print("Videojuegos procesados:",
      cantidad)


print("\nTop juegos recomendados:")

dfs_recommendation.show(10)


fin = time.time()

print("\nTiempo de ejecución:",
      fin-inicio,
      "segundos")
tiempos_resultados["Spark_Consulta_7"] = fin - inicio


===== SPARK =====


Videojuegos procesados: 22992

Top juegos recomendados:


+--------------------+-------------+----------------+-------------------------+
|                game|total_reviews|positive_reviews|recommendation_percentage|
+--------------------+-------------+----------------+-------------------------+
|Library Of Ruina ...|            5|               5|                    100.0|
|Welcome to Free W...|            1|               1|                    100.0|
|Eve of Souls: Sta...|            3|               3|                    100.0|
|       Summer of '58|           15|              15|                    100.0|
|          Bus Driver|            9|               9|                    100.0|
|            TRON 2.0|            7|               7|                    100.0|
|        Bad Memories|            1|               1|                    100.0|
|             80 Days|            9|               9|                    100.0|
|Outcore: Clown No...|            3|               3|                    100.0|
|             Obscure|            7|    

## Consulta 8 - Promedio de horas jugadas por videojuego

Objetivo: calcular promedio de author_playtime_forever por juego.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [11]:
# ==========================================================
# CONSULTA 8 - PROMEDIO DE HORAS JUGADAS POR VIDEOJUEGO
# SPARK
# ==========================================================

import time

from pyspark.sql.functions import (
    avg,
    desc
)


inicio = time.time()

print("===== SPARK =====")


dfs_playtime = (
    dfs_transform
    .groupBy("game")
    .agg(
        avg(
            "author_playtime_forever"
        )
        .alias(
            "avg_playtime_minutes"
        )
    )
    .orderBy(
        desc(
            "avg_playtime_minutes"
        )
    )
)


# Ejecutar Spark

cantidad = dfs_playtime.count()


print("Videojuegos procesados:",
      cantidad)


print("\nVideojuegos con mayor promedio de juego:")

dfs_playtime.show(10)



fin = time.time()

print("\nTiempo de ejecución:",
      fin-inicio,
      "segundos")
tiempos_resultados["Spark_Consulta_8"] = fin - inicio


===== SPARK =====


Videojuegos procesados: 22992

Videojuegos con mayor promedio de juego:


+--------------------+--------------------+
|                game|avg_playtime_minutes|
+--------------------+--------------------+
|Oops!!! I Slept W...|           2181697.0|
| Legions of Ashworld|           1982834.0|
|  Aquarium Simulator|           1687347.0|
|            WalkinVR|           1366674.0|
|       Houdini Indie|           1264521.5|
|Oh, you touch my ...|           1247696.0|
|The Putinland: Di...|           1114523.0|
|        MachineCraft|            987286.0|
|Solitaire Forever II|            945538.0|
|Crusaders of the ...|            903523.0|
+--------------------+--------------------+
only showing top 10 rows


Tiempo de ejecución: 32.32019925117493 segundos


## Consulta 9 - Longitud promedio de reseñas por videojuego

Objetivo: calcular promedio de review_length agrupado por game.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [12]:
# ==========================================================
# CONSULTA 9 - LONGITUD PROMEDIO DE RESEÑAS POR VIDEOJUEGO
# SPARK
# ==========================================================

import time

from pyspark.sql.functions import (
    avg,
    desc
)


inicio = time.time()

print("===== SPARK =====")


dfs_review_length = (
    dfs_transform
    .groupBy("game")
    .agg(
        avg(
            "review_length"
        )
        .alias(
            "avg_review_length"
        )
    )
    .orderBy(
        desc(
            "avg_review_length"
        )
    )
)


# Ejecutar Spark

cantidad = dfs_review_length.count()


print("Videojuegos procesados:",
      cantidad)


print("\nVideojuegos con reseñas más extensas:")

dfs_review_length.show(10)



fin = time.time()

print("\nTiempo de ejecución:",
      fin - inicio,
      "segundos")
tiempos_resultados["Spark_Consulta_9"] = fin - inicio


===== SPARK =====


Videojuegos procesados: 22992

Videojuegos con reseñas más extensas:


+--------------------+-----------------+
|                game|avg_review_length|
+--------------------+-----------------+
|      Mary Skelter 2|           8000.0|
|      Wayward Strand|           8000.0|
|   Indie Game Battle|           8000.0|
|Qbeh-1: The Atlas...|           8000.0|
|    The Knight Witch|           8000.0|
|        Azusa Online|           7999.0|
|        Deadly Flare|           7999.0|
|      Alterium Shift|           7998.0|
|            Dynopunk|           7996.0|
|       The Companion|           7996.0|
+--------------------+-----------------+
only showing top 10 rows


Tiempo de ejecución: 32.30847907066345 segundos


## Consulta 10 - Ranking de videojuegos

Objetivo: ordenar videojuegos considerando porcentaje de recomendación y cantidad de reseñas.

### Implementación

Se desarrollará la misma lógica en:

- Polars
- Dask
- Modin
- Spark

### Resultados

Registrar:
- cantidad de registros procesados;
- tiempo de ejecución;
- observaciones del procesamiento distribuido.


In [13]:
# ==========================================================
# CONSULTA 10 - RANKING DE VIDEOJUEGOS
# SPARK
# ==========================================================

import time

from pyspark.sql.functions import (
    count,
    sum,
    col,
    desc
)


inicio = time.time()

print("===== SPARK =====")


dfs_ranking = (
    dfs_transform
    .groupBy("game")
    .agg(
        count("*")
        .alias("total_reviews"),

        sum("voted_up")
        .alias("positive_reviews")
    )
    .withColumn(
        "recommendation_percentage",
        (
            col("positive_reviews") /
            col("total_reviews")
            * 100
        )
    )
    .filter(
        col("total_reviews") >= 100
    )
    .orderBy(
        [
            desc("recommendation_percentage"),
            desc("total_reviews")
        ]
    )
)


# Ejecutar Spark

cantidad = dfs_ranking.count()


print("Videojuegos rankeados:",
      cantidad)


print("\nTop videojuegos:")

dfs_ranking.show(10)


fin = time.time()

print("\nTiempo de ejecución:",
      fin - inicio,
      "segundos")
tiempos_resultados["Spark_Consulta_10"] = fin - inicio


===== SPARK =====


Videojuegos rankeados: 609

Top videojuegos:


+--------------------+-------------+----------------+-------------------------+
|                game|total_reviews|positive_reviews|recommendation_percentage|
+--------------------+-------------+----------------+-------------------------+
|       Left 4 Dead 2|          821|             821|                    100.0|
|          Subnautica|          807|             807|                    100.0|
|              Mirror|          604|             604|                    100.0|
|Plants vs. Zombie...|          532|             532|                    100.0|
|Mount & Blade: Wa...|          428|             428|                    100.0|
|               Hades|          390|             390|                    100.0|
|      Counter-Strike|          386|             386|                    100.0|
|        Satisfactory|          384|             384|                    100.0|
|                DOOM|          382|             382|                    100.0|
|      Papers, Please|          356|    

In [14]:
# ==========================================================
# EXPORTACION AUTOMATIZADA DE RESULTADOS A GOOGLE CLOUD STORAGE
# ==========================================================

!pip install -q gcsfs fsspec

import pandas as pd

framework = "spark"

df_tiempos = pd.DataFrame(
    list(tiempos_resultados.items()),
    columns=["Consulta", "Tiempo_segundos"]
)

print(df_tiempos)

ruta_salida = f"gs://bigdata-2026-02/proyecto01/tiempos_{framework}_{arquitectura}.csv"

df_tiempos.to_csv(ruta_salida, index=False)

print(f"\nResultados exportados a: {ruta_salida}")


            Consulta  Tiempo_segundos
0   Spark_Consulta_1        24.144484
1   Spark_Consulta_2        27.441231
2   Spark_Consulta_3        81.976911
3   Spark_Consulta_4        32.519988
4   Spark_Consulta_5        46.275426
5   Spark_Consulta_6        34.418114
6   Spark_Consulta_7        32.725953
7   Spark_Consulta_8        32.320199
8   Spark_Consulta_9        32.308479
9  Spark_Consulta_10        32.841287

Resultados exportados a: gs://bigdata-2026-02/proyecto01/tiempos_spark_2_workers.csv
